<div class="alert alert-info">
    <b>2.4.1 Extreme Ion Cases Analysis</b><br>
    Dieses Notebook analysiert Extremfälle des Ionenbilanzfehlers (IBE).<br>
    -> <b>Ziel:</b> Erstellung einer PDF ('Extreme_IBE_Cases.pdf') mit Beispielen für IBEs von +/- 100%, 75%, 50%, 25%, 10% und 5%.<br>
    -> <b>Output:</b> PDF-Datei mit je 30 Beispielen pro Fehlerklasse.
</div>

In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

In [ ]:
# ------------------------- Einstellungen -------------------------
TARGET_PERCENTAGES = [100, 75, 50, 25, 10, 5]
SAMPLES_PER_PAGE = 30
REQUIRED_IONS = ['Na', 'Ca', 'Mg', 'Cl', 'HCO3', 'SO4']

# ------------------------- Valenz-Wörterbuch zur Normalisierung -------------------------
ION_SPECS = {
    'Na':   {'valence': 1, 'mass': 22.99, 'type': 'cation', 'regex': r'Na_in_([a-zA-Z0-9/]+)'},
    'Ca':   {'valence': 2, 'mass': 40.08, 'type': 'cation', 'regex': r'Ca_in_([a-zA-Z0-9/]+)'},
    'Mg':   {'valence': 2, 'mass': 24.31, 'type': 'cation', 'regex': r'Mg_in_([a-zA-Z0-9/]+)'},
    'Cl':   {'valence': 1, 'mass': 35.45, 'type': 'anion',  'regex': r'Cl_in_([a-zA-Z0-9/]+)'},
    'HCO3': {'valence': 1, 'mass': 61.02, 'type': 'anion',  'regex': r'HCO3_in_([a-zA-Z0-9/]+)'},
    'SO4':  {'valence': 2, 'mass': 96.06, 'type': 'anion',  'regex': r'SO4_in_([a-zA-Z0-9/]+)'},
}

<div class="alert alert-info">
    <b>1. Daten laden und IBE berechnen</b>
</div>

In [ ]:
def get_conversion_factor(unit, valence, mass):
    # ------------------------- Gibt den Umrechnungsfaktor zu meq/L zurück -------------------------
    unit = unit.lower()
    if unit == 'mmol/l':
        return valence
    elif unit == 'mg/l':
        return valence / mass
    elif unit == 'meq/l':
        return 1.0
    elif unit == 'ppm':
        return valence / mass
    return 0.0

def calculate_ibe(df):
    # ------------------------- Berechnet den Ladungsbilanzfehler (Charge Balance Error) NUR für die definierten 6 Hauptionen -------------------------
    cations_sum = pd.Series(0.0, index=df.index)
    anions_sum = pd.Series(0.0, index=df.index)
    
    for ion, specs in ION_SPECS.items():
        regex = specs['regex']
        matches = [c for c in df.columns if re.search(regex, c)]
        
        if matches:
            col_name = matches[0]
            unit_match = re.search(regex, col_name)
            if unit_match:
                unit = unit_match.group(1)
                factor = get_conversion_factor(unit, specs['valence'], specs['mass'])
                vals = pd.to_numeric(df[col_name], errors='coerce').fillna(0)
                meq_vals = vals * factor
                
                if specs['type'] == 'cation':
                    cations_sum += meq_vals
                else:
                    anions_sum += meq_vals
    
    total_ions = cations_sum + anions_sum
    ibe = pd.Series(np.nan, index=df.index)
    valid_mask = total_ions > 0.0001
    ibe[valid_mask] = ((cations_sum[valid_mask] - anions_sum[valid_mask]) / total_ions[valid_mask]) * 100
    
    return ibe, valid_mask

In [ ]:
# ------------------------- Datenbankpfad finden -------------------------
base_dir = Path.cwd()
notebooks_root = base_dir.parent
db_archive_path = notebooks_root / "1.1_Data-Acquisition-Wrapper" / "Angepasste_Datenbanken" / "Nach_Acquisition" / "Komplette_Datenbank"

# ------------------------- Suche nach der aktuellsten Datenbank -------------------------
all_folders = sorted([f for f in db_archive_path.iterdir() if f.is_dir()], key=lambda x: x.name, reverse=True)

input_csv = None
for folder in all_folders:
    csv_path = folder / "Komplette_Datenbank.csv"
    if csv_path.exists() and csv_path.stat().st_size > 10 * 1024 * 1024:
        input_csv = csv_path
        break

if input_csv is None:
    raise FileNotFoundError("Keine Datenbank > 10MB gefunden! Bitte prüfen Sie den Pfad.")

print(f"Lade Datenbank: {input_csv}")
df = pd.read_csv(input_csv, low_memory=False)

print("Berechne IBE...")
ibe_values, valid_mask = calculate_ibe(df)
df['IBE'] = ibe_values

# ------------------------- Nur Zeilen mit berechnetem IBE behalten -------------------------
df = df[valid_mask].copy()

# ------------------------- Spalten Inspektion -------------------------
print("\n--- Spalten Inspektion ---")
print("Anzahl Spalten:", len(df.columns))
print("Spalten:", df.columns.tolist())
display(df.head())

<div class="alert alert-info">
    <b>2. Extremfälle extrahieren und PDF erstellen</b>
</div>

In [ ]:
def get_extreme_examples(df, target_percent, n=30):

    n_half = n // 2
    
    # ------------------------- Hilfsfunktion zum Finden der besten Matches pro Gruppe -------------------------
    def get_best_per_source(subset, target):
        diff = (subset['IBE'] - target).abs()
        # ------------------------- Temporär Differenz speichern -------------------------
        temp_df = subset.copy()
        temp_df['diff_to_target'] = diff
        
        # ------------------------- Nach Differenz sortieren -------------------------
        temp_df = temp_df.sort_values('diff_to_target')
        
        # ------------------------- Erste Vorkommen pro Datenbank_name behalten -------------------------
        # ------------------------- Falls 'Database_name' nicht existiert, Fallback auf alles -------------------------
        if 'Database_name' in temp_df.columns:
            unique_db = temp_df.drop_duplicates(subset=['Database_name'], keep='first')
            return unique_db
        return temp_df

    # ------------------------- Positive Abweichung (Kationen > Anionen) -------------------------
    closest_pos = get_best_per_source(df[df['IBE'] > 0], target_percent)
    closest_pos = closest_pos.head(n_half)
    
    # ------------------------- Negative Abweichung (Anionen > Kationen) -------------------------
    closest_neg = get_best_per_source(df[df['IBE'] < 0], -target_percent)
    closest_neg = closest_neg.head(n_half)
    
    result = pd.concat([closest_pos, closest_neg])
    # ------------------------- Sortieren nach IBE für schönere Darstellung -------------------------
    result = result.sort_values(by='IBE', ascending=False)
    return result

def create_table_page(pdf, df_subset, target):
    # ------------------------- Erstellt eine Seite im PDF mit der Tabelle -------------------------
    fig, ax = plt.subplots(figsize=(11.69, 8.27)) # A4 Landscape
    ax.axis('tight')
    ax.axis('off')
    ax.set_title(f"Beispiele für ca. +/- {target}% Ionenbilanzfehler", fontsize=16, weight='bold', pad=20)
    
    # ------------------------- Spalten auswählen -------------------------
    df_display = df_subset.reset_index().rename(columns={'index': 'Orig. Row'})
    
    cols_to_show = ['Orig. Row']
    if 'Database_name' in df_subset.columns:
        cols_to_show.append('Database_name')
        
    cols_to_show.extend(['IBE', 'temperature_in_c'])
    
    # ------------------------- Ionen-Spalten finden -------------------------
    for ion in REQUIRED_IONS:
        specs = ION_SPECS[ion]
        matches = [c for c in df_subset.columns if re.search(specs['regex'], c)]
        if matches:
            cols_to_show.append(matches[0])
            
    # ------------------------- Daten vorbereiten -------------------------
    table_data = df_display[cols_to_show].copy()
    
    # ------------------------- Runden für bessere Lesbarkeit -------------------------
    for col in table_data.select_dtypes(include=['float']).columns:
        table_data[col] = table_data[col].apply(lambda x: f"{x:.2f}")
        
    # ------------------------- Datenbanknamen kürzen, falls zu lang -------------------------
    if 'Database_name' in table_data.columns:
        table_data['Database_name'] = table_data['Database_name'].apply(lambda x: (str(x)[:15] + '..') if len(str(x)) > 15 else str(x))

    # ------------------------- Spaltennamen kürzen für Darstellung -------------------------
    short_cols = [c.replace('_in_mg/l', '').replace('_in_mmol/l', '*').replace('temperature_in_c', 'Temp').replace('Database_name', 'DB') for c in cols_to_show]
    
    table = ax.table(cellText=table_data.values, colLabels=short_cols, loc='center', cellLoc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(8)
    table.scale(1.2, 1.2)
    
    pdf.savefig(fig, bbox_inches='tight')
    plt.close()

In [ ]:
output_pdf = "Extreme_IBE_Cases.pdf"

with PdfPages(output_pdf) as pdf:
    for target in TARGET_PERCENTAGES:
        print(f"Bearbeite {target}%...")
        subset = get_extreme_examples(df, target, SAMPLES_PER_PAGE)
        create_table_page(pdf, subset, target)
        
print(f"Fertig! PDF gespeichert unter: {Path(output_pdf).absolute()}")